In [2]:
import xarray as xr
from ipywidgets import interact, IntSlider
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
sns.set_theme()

In [3]:
def restrict_dataset_to_hypercube(paramsDataset, metricsDataset, threshold, non_restricted_param_idx):


    ctrl_params = paramsDataset.where(paramsDataset.ens_idx.str.match('ctrl'), drop= True).params.values[0,:-1]

    min_vals = np.min(paramsDataset.params.values,axis =0)
    max_vals = np.max(paramsDataset.params.values,axis =0)
    ranges = max_vals - min_vals

    threshold = np.repeat(threshold,len(min_vals))

    threshold[non_restricted_param_idx]  = 1

    print(threshold)

    lower_bounds = min_vals + ((1-threshold)/2) * ranges
    upper_bounds = max_vals - ((1-threshold)/2) * ranges

    lower_accepted = np.all( lower_bounds <= paramsDataset.params.values,axis = 1)
    upper_accepted = np.all(paramsDataset.params.values <= upper_bounds, axis = 1)


    indices = np.where(lower_accepted & upper_accepted)[0]

    print(f"Number of PPE runs before restriction: {paramsDataset.ens_idx.size}")
    print(f"Number of PPE runs after restriction: {indices.size}")

    restricted_paramsDataset = paramsDataset.isel(ens_idx=indices)
    restricted_metricsDataset = metricsDataset.isel(ens_idx=indices)

    return restricted_paramsDataset, restricted_metricsDataset, indices

In [4]:
def restrict_dataset_by_param(params_dataset, metrics_dataset, param_index, lower_bound):

    ctrl_params =  params_dataset.where(params_dataset.ens_idx.str.match('ctrl'), drop= True).params.values.flatten()


    if ctrl_params[param_index] <= lower_bound:
        raise ValueError("The lower bound should be lower than the control parameter.")


    indices = np.where(params_dataset.params[:,param_index] > lower_bound)[0]

    restricted_paramsDataset = params_dataset.isel(ens_idx=indices)
    restricted_metricsDataset = metrics_dataset.isel(ens_idx=indices)

    print(indices.shape)

    return restricted_paramsDataset, restricted_metricsDataset, indices

In [5]:
def restrict_dataset_by_metric_loss(params_dataset,metrics_dataset,varPrefixes,boxSize, threshold):



    metric_names = []
    for varPrefix in varPrefixes:
        for i in range(1,int(180/boxSize)+1):
            for j in range(1,int(360/boxSize)+1):
                metric_names.append(f"{varPrefix}_{i}_{j}")

    default = metrics_dataset.where(metrics_dataset.ens_idx.str.match('ctrl'), drop= True)[metric_names].isel(time=0,product=0,ens_idx=0).to_array().values

    

    metric_data = metrics_dataset[metric_names].isel(time=0,product=0).to_array().values

    losses = np.sum((default - metric_data.T)**2, axis=1)

    

    quantile = np.percentile(losses, 100*threshold)

    indices = np.where(losses <= quantile)[0]

    bad_indices = np.where(losses > quantile)[0]
    # print(bad_indices)

    print(f"Number of PPE runs before restriction: {params_dataset.ens_idx.size}")
    print(f"Number of PPE runs after restriction: {indices.size}")

    restricted_paramsDataset = params_dataset.isel(ens_idx=indices)
    restricted_metricsDataset = metrics_dataset.isel(ens_idx=indices)
        
    return restricted_paramsDataset, restricted_metricsDataset, indices

In [6]:
# metrics_dataset = xr.open_dataset("./tuning_files/PPE_Data/20H003_rshp_w_obs_Regional.nc",engine="netcdf4")  
metrics_dataset = xr.open_dataset("./tuning_files/PPE_Data/PPE_7_5deg_metrics.nc",engine="netcdf4")  
params_dataset = xr.open_dataset('./tuning_files/PPE_Data/H003_rshp_w_obs.nc',engine="netcdf4") 



# restricted_params_dataset, restricted_metrics_dataset,indices = restrict_dataset_to_hypercube('./tuning_files/PPE_Data/H003_rshp_w_obs.nc', "./tuning_files/PPE_Data/PPE_7_5deg_metrics.nc", 0.9, 1)


In [7]:


restricted_params_dataset, restricted_metrics_dataset, indices   = restrict_dataset_by_param(params_dataset, metrics_dataset, -1, 20e4)

restricted_params_dataset, restricted_metrics_dataset, indices = restrict_dataset_by_metric_loss(restricted_params_dataset, restricted_metrics_dataset,["SWCF"],7.5,0.3)

(93,)
Number of PPE runs before restriction: 93
Number of PPE runs after restriction: 28


In [8]:

from scipy import stats


paramNames = ['clubb_c1',
        'clubb_gamma_coef',
        'zmconv_tau',
        'zmconv_dmpdz',
        'zmconv_micro_dcs',
        'nucleate_ice_subgrid',
        'p3_nc_autocon_expon',
        'p3_qc_accret_expon',
        'zmconv_auto_fac',
        'zmconv_accr_fac',
        'zmconv_ke',
        'cldfrc_dp1',
        'p3_embryonic_rain_size',
        'p3_mincdnc'
        ]

def get_swcf_data(x,y,restricted=False):

    ds = restricted_metrics_dataset if restricted else metrics_dataset

    return ds[f"SWCF_{y}_{x}"].isel(time=0,product=0).values




def plot_interactive_plot(x,y,param,order=2):

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    data = get_swcf_data(x,y)
    params = params_dataset.params.values[:,param]
    sns.regplot(x=params, y=data, order=order, scatter=True, line_kws={'color': 'red'},ci=None,fit_reg=True, ax=axes[0])

    restricted_params = restricted_params_dataset.params.values[:,param]
    restricted_data = get_swcf_data(x,y,restricted=True)


    sns.regplot(x=restricted_params, y=restricted_data, order=order, scatter=True,scatter_kws={'alpha':0}, line_kws={'color': 'blue'},ci=None,fit_reg=True, ax=axes[1])
    _,_,r_val,_,_  = stats.linregress(restricted_params, restricted_data)

    plt.title(f"R-squared value for restricted dataset: {r_val**2:.3f}")
    for idx, (x_val,y_val) in enumerate(zip(restricted_params, restricted_data)):
        axes[1].text(x_val,y_val, indices[idx], fontsize=9,ha='center',va='center')


    
    plt.xlabel(paramNames[param])
    plt.ylabel("metric_value")
    plt.show()

interact(plot_interactive_plot,
         x=IntSlider(min=1,max=48,step=1,value=36),
         y=IntSlider(min=1,max=24,step=1,value=15),
         param=IntSlider(min=0,max=12,step=1,value=12))



interactive(children=(IntSlider(value=36, description='x', max=48, min=1), IntSlider(value=15, description='y'…

<function __main__.plot_interactive_plot(x, y, param, order=2)>

In [13]:

paramNames = ['clubb_c1',
        'clubb_gamma_coef',
        'zmconv_tau',
        'zmconv_dmpdz',
        'zmconv_micro_dcs',
        'nucleate_ice_subgrid',
        'p3_nc_autocon_expon',
        'p3_qc_accret_expon',
        'zmconv_auto_fac',
        'zmconv_accr_fac',
        'zmconv_ke',
        'cldfrc_dp1',
        'p3_embryonic_rain_size',
        'p3_mincdnc'
        ]

def get_swcf_data(x,y):
    metrics = []
    for member in range(restricted_metrics_dataset.ens_idx.size):
        metrics.append(restricted_metrics_dataset.isel(time=0,product=0,ens_idx=member)[f"SWCF_{y}_{x}"].values)

    return np.array(metrics)



def plot_interactive_plot(x,y,param,order=1):
    data = get_swcf_data(x,y,restricted=True)
    params = restricted_params_dataset.params.values[:,param]
    sns.regplot(x=params, y=data, order=order, scatter=True, line_kws={'color': 'red'},ci=None,fit_reg=True)
    plt.xlabel(paramNames[param])
    plt.ylabel("metric_value")
    plt.show()

interact(plot_interactive_plot,
         x=IntSlider(min=1,max=48,step=1,value=2),
         y=IntSlider(min=1,max=24,step=1,value=9),
         param=IntSlider(min=0,max=12,step=1,value=0))



interactive(children=(IntSlider(value=2, description='x', max=48, min=1), IntSlider(value=9, description='y', …

<function __main__.plot_interactive_plot(x, y, param, order=1)>

In [1]:
varprefix = "SWCF" 
metrics = []

for i in range(1,10): #TODO dont have this 10 fixed
    for j in range(1,19):#TODO dont have this 19 fixed
        metrics.append(f"{varprefix}_{i}_{j}")

paramNames = ['clubb_c1',
        'clubb_gamma_coef',
        'zmconv_tau',
        'zmconv_dmpdz',
        'zmconv_micro_dcs',
        'zmconv_auto_fac',
        'zmconv_accr_fac',
        'zmconv_ke',
        'nucleate_ice_subgrid',
        'p3_nc_autocon_expon',
        'p3_qc_accret_expon',
        'cldfrc_dp1',
        'p3_embryonic_rain_size',
        'p3_mincdnc']

def get_ppe_data(ppe_index):
    data = []
    for member in metrics:
        data.append(metrics_dataset.isel(time=0,product=0,ens_idx=ppe_index)[f"{member}"].values)

    return np.array(data)



def plot_interactive_plot(ppe_index):
    data = get_ppe_data(ppe_index)

    grid_data = data.reshape((9,18))



    plt.imshow(grid_data, origin='upper')
    plt.colorbar()
    plt.show()

interact(plot_interactive_plot,
         ppe_index=IntSlider(min=0,max=431,step=1,value=0))



NameError: name 'interact' is not defined